# Atlas Python Static Analysis Tutorial

Atlas builds a complete navigable tree of your Python codebase through AST-based reconnaissance.

## 1. Basic Setup

Build a project tree by pointing Atlas at your target directory:

In [1]:
from analyzer import build_complete_atlas

# Build complete project tree
project = build_complete_atlas('sample_files')
print(f"Project: {project.name}")

Project: sample_files


## 2. Tree Visualization

View the complete hierarchical structure:

In [2]:
# Print entire tree to console
project.print()

Project(sample_files)
  Package(api)
    Package(endpoints)
      Module(product_endpoints)
        Class(ProductEndpoints)
          Function(__init__)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
          Function(create_category)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(name)
            Argument(description)
              MissingArgumentTypeHint(ArgumentNode)
          Function(get_category)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(category_id)
              MissingArgumentTypeHint(ArgumentNode)
          Function(create_product)
            Argument(self)
              MissingArgumentTypeHint(ArgumentNode)
            Argument(name)
            Argument(price)
            Argument(category_id)
            Argument(description)
              MissingArgumentTypeHint(ArgumentNode)
          Function(get_product)
            Argument(

## 3. Navigation API

Navigate through packages, modules, classes, and functions:

In [3]:
# List all packages
packages = project.list_packages()
print(f"Found {len(packages)} packages")

# Get specific package
models_pkg = project.get_package('models')
print(f"Package: {models_pkg.fqn}")

# List modules in package
modules = models_pkg.list_modules()
print(f"Modules: {[m.name for m in modules]}")

Found 5 packages
Package: sample_files.models
Modules: ['order', 'product', 'user']


## 4. Class Discovery

Explore classes and their methods:

In [4]:
# Get a module and list its classes
user_module = models_pkg.get_module('user')
classes = user_module.list_classes()
print(f"Classes in {user_module.name}: {[c.name for c in classes]}")

# Get specific class
user_class = user_module.get_class('User')
print(f"\nClass: {user_class.fqn}")

# List methods
methods = user_class.list_methods()
print(f"Methods: {[m.name for m in methods]}")

Classes in user: ['User', 'UserProfile']

Class: sample_files.models.user.User
Methods: ['__init__', 'get_email', 'set_email', 'add_role', 'has_role', 'get_roles', 'activate', 'deactivate']


## 5. Function Arguments and Return Types

Examine function signatures and type information:

In [5]:
# Get a method and examine its signature
add_role = user_class.get_method('add_role')
print(f"Method: {add_role.name}")

# List arguments and check for type information
args = add_role.list_arguments()
for arg in args:
    # Check if argument has type annotation via _violations attribute
    if hasattr(arg, '_violations') and arg._violations:
        print(f"  {arg.name}: (missing type hint)")
    else:
        # Has type - access through _type attribute
        if hasattr(arg, '_type') and arg._type:
            print(f"  {arg.name}: {arg._type.name}")

# Check return type
returns = add_role.list_returns()
if returns:
    return_node = returns[0]
    if hasattr(return_node, '_violations') and return_node._violations:
        print(f"Returns: (missing type hint)")
    else:
        if hasattr(return_node, '_type') and return_node._type:
            print(f"Returns: {return_node._type.name}")

Method: add_role
  self: (missing type hint)
  role: str


## 6. Attribute Discovery

Find class-level and instance attributes:

In [6]:
# Class-level attributes
class_attrs = user_class.list_class_attributes()
print(f"Class attributes: {[a.name for a in class_attrs]}")

# Instance attributes (from __init__)
instance_attrs = user_class.list_instance_attributes()
print(f"Instance attributes: {[a.name for a in instance_attrs]}")

Class attributes: []
Instance attributes: ['email', 'username', 'password', 'is_active', 'roles']


## 7. Import Analysis

Discover module imports:

In [7]:
# List all imports in a module
imports = user_module.list_imports()
print(f"Found {len(imports)} import statements")

# Examine individual imports
for imp in imports[:3]:  # Show first 3
    aliases = imp.list_aliases()
    for alias in aliases:
        print(f"  {alias.local_name} -> {alias.original_name}")

Found 3 import statements


AttributeError: 'AliasNode' object has no attribute 'local_name'

## 8. Violation Detection

Atlas automatically detects missing type hints via violation ornaments:

In [8]:
# Check for violations on arguments
method = user_class.get_method('set_email')
for arg in method.list_arguments():
    # Access _violations attribute directly
    if hasattr(arg, '_violations') and arg._violations:
        print(f"  {arg.fqn}: {arg._violations[0].__class__.__name__}")

# Check for violations on return types
returns = method.list_returns()
if returns:
    return_node = returns[0]
    if hasattr(return_node, '_violations') and return_node._violations:
        print(f"  Return type: {return_node._violations[0].__class__.__name__}")

  sample_files.models.user.User.set_email.self: MissingArgumentTypeHint
  sample_files.models.user.User.set_email.email: MissingArgumentTypeHint


## 9. Recursive Discovery

Find all entities of a specific type throughout the tree:

In [9]:
# Find all classes in entire project
all_classes = project.list_all_classes()
print(f"Total classes: {len(all_classes)}")

# Find all functions (module-level)
all_functions = project.list_all_functions()
print(f"Total module-level functions: {len(all_functions)}")

# Find all methods
all_methods = project.list_all_methods()
print(f"Total methods: {len(all_methods)}")

Total classes: 27
Total module-level functions: 15
Total methods: 142


## 10. JSON Serialization

Export the entire project tree as JSON:

In [10]:
# Get JSON as dict
data = project.dump()
print(f"Serialized {len(data['children'])} top-level packages")

# Save to file
project.save_dump('project_analysis.json', indent=2)

Serialized 5 top-level packages
✓ Project tree serialized to: project_analysis.json
  File size: 78,106 bytes
  Nodes serialized: 246


## 11. Practical Example: Find All Type Violations

Scan the entire project for missing type hints:

In [11]:
# Count violations by type
arg_violations = 0
return_violations = 0

# Check all methods
for method in project.list_all_methods():
    # Check arguments - access _violations directly
    for arg in method.list_arguments():
        if hasattr(arg, '_violations') and arg._violations:
            arg_violations += 1
    
    # Check return type
    returns = method.list_returns()
    if returns:
        return_node = returns[0]
        if hasattr(return_node, '_violations') and return_node._violations:
            return_violations += 1

print(f"Missing argument type hints: {arg_violations}")
print(f"Missing return type hints: {return_violations}")
print(f"Total violations: {arg_violations + return_violations}")

Missing argument type hints: 226
Missing return type hints: 0
Total violations: 226


## 12. Practical Example: Analyze Method Complexity

Find methods with many parameters:

In [12]:
# Find methods with 4+ parameters
complex_methods = []

for method in project.list_all_methods():
    args = method.list_arguments()
    # Exclude 'self' from count
    param_count = len([a for a in args if a.name != 'self'])
    if param_count >= 4:
        complex_methods.append((method.fqn, param_count))

print(f"Methods with 4+ parameters: {len(complex_methods)}")
for fqn, count in sorted(complex_methods, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {fqn}: {count} parameters")

Methods with 4+ parameters: 10
  sample_files.models.product.Product.__init__: 5 parameters
  sample_files.services.email_service.EmailService.send_email: 5 parameters
  sample_files.services.payment_service.PaymentProcessor.process_payment: 5 parameters
  sample_files.api.endpoints.product_endpoints.ProductEndpoints.create_product: 4 parameters
  sample_files.models.user.User.__init__: 4 parameters
